# ADRE-CXR — LoRA Fine-tune (Kaggle GPU)

**Goal:** Fine-tune a small LLM with LoRA on IU X-Ray reports, then generate Findings+Impression for the same 26 eval UIDs.

**Attach:** `raddar/chest-xrays-indiana-university`  
**Optional:** upload `heldout_uids.json`  
**Settings:** GPU T4, Internet On

**Note:** Llama-3.2 is gated. This notebook defaults to open **Qwen2.5-1.5B-Instruct**.

Download `lora_predictions.jsonl` → `adre-cxr/outputs/` then locally:
```powershell
python scripts/evaluate.py --predictions outputs/lora_predictions.jsonl --split dev
python scripts/build_comparison_table.py
```

In [ ]:
# --- 1. Install dependencies ---
# After this cell: Runtime → Restart session, then run from cell 2.
!pip uninstall -y bitsandbytes -q
!pip install -q "transformers==4.46.3" "peft==0.13.2" "trl==0.11.4" "accelerate==1.2.1" "datasets==3.2.0" sentencepiece protobuf
!pip install -q --upgrade bitsandbytes
print("Install done. Restart session now, then continue from cell 2.")

In [ ]:
# --- 2. Paths ---
from pathlib import Path
import os
import json

CANDIDATES = [
    Path("/kaggle/input/chest-xrays-indiana-university"),
    Path("/kaggle/input/datasets/raddar/chest-xrays-indiana-university"),
    Path("/kaggle/input/raddar/chest-xrays-indiana-university"),
]
DATA_ROOT = None
for p in CANDIDATES:
    if (p / "indiana_reports.csv").exists():
        DATA_ROOT = p
        break
if DATA_ROOT is None:
    for p in Path("/kaggle/input").rglob("indiana_reports.csv"):
        DATA_ROOT = p.parent
        break
if DATA_ROOT is None:
    raise FileNotFoundError("Attach dataset: raddar/chest-xrays-indiana-university")

WORK = Path("/kaggle/working")
ADAPTER_DIR = WORK / "lora_adapter"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

HELDOUT = None
for p in Path("/kaggle/input").rglob("heldout_uids.json"):
    HELDOUT = json.loads(p.read_text(encoding="utf-8"))
    print("Loaded heldout_uids from", p)
    break

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print("HF login OK")
else:
    print("No HF_TOKEN — using open Qwen model (Llama gated)")

print("DATA_ROOT:", DATA_ROOT)

In [ ]:
# --- 3. Build train / eval splits ---
import pandas as pd
import numpy as np

MIN_FINDINGS, MIN_IMPRESSION = 10, 5
DEV_SIZE, TEST_SIZE, SEED = 100, 400, 42

df = pd.read_csv(DATA_ROOT / "indiana_reports.csv")
df["uid"] = df["uid"].astype(str)
for col in ("findings", "impression", "indication"):
    df[col] = df[col].fillna("").astype(str)

usable = df[(df["findings"].str.len() >= MIN_FINDINGS) & (df["impression"].str.len() >= MIN_IMPRESSION)].copy()

if HELDOUT is not None:
    held = set(map(str, HELDOUT["dev"])) | set(map(str, HELDOUT["test"]))
    eval_uids = [str(u) for u in HELDOUT.get("eval_uids", HELDOUT["dev"][:26])]
else:
    rng = np.random.default_rng(SEED)
    uids = usable["uid"].tolist()
    rng.shuffle(uids)
    held = set(uids[: DEV_SIZE + TEST_SIZE])
    eval_uids = uids[:26]

train_df = usable[~usable["uid"].isin(held)].copy()
eval_df = usable[usable["uid"].isin(eval_uids)].copy()
eval_df["_ord"] = eval_df["uid"].map({u: i for i, u in enumerate(eval_uids)})
eval_df = eval_df.sort_values("_ord").drop(columns="_ord")

print(f"Train rows: {len(train_df)}")
print(f"Eval rows:  {len(eval_df)} (target 26)")
print("Eval UIDs:", eval_df["uid"].tolist())

In [ ]:
# --- 4. Format chat-style SFT examples ---
from datasets import Dataset

INSTRUCTION = (
    "You are a radiologist. Given the clinical indication, write a chest X-ray report "
    "with exactly two sections: FINDINGS and IMPRESSION."
)

def to_target(findings: str, impression: str) -> str:
    return f"FINDINGS:\n{findings.strip()}\n\nIMPRESSION:\n{impression.strip()}"

def build_messages(indication: str, findings: str, impression: str):
    return [
        {"role": "system", "content": INSTRUCTION},
        {"role": "user", "content": f"Indication: {indication.strip() or 'Not provided'}"},
        {"role": "assistant", "content": to_target(findings, impression)},
    ]

train_rows = [
    {"messages": build_messages(r["indication"], r["findings"], r["impression"])}
    for _, r in train_df.iterrows()
]
train_ds = Dataset.from_list(train_rows)
print(train_ds)
print(train_ds[0]["messages"][1]["content"][:120])

In [ ]:
# --- 5. Load base model ---
# Prefer open Qwen. Try 4-bit; if bitsandbytes is broken on this Kaggle image, use fp16.
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

for name in ("model", "trainer"):
    if name in globals():
        try:
            del globals()[name]
        except Exception:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CANDIDATE_MODELS = [
    "Qwen/Qwen2.5-1.5B-Instruct",  # open — use this
    "meta-llama/Llama-3.2-1B-Instruct",  # gated — needs HF_TOKEN + license accept
]

def load_tokenizer(mid):
    tok = AutoTokenizer.from_pretrained(mid, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    return tok

def try_load(mid, use_4bit: bool):
    tok = load_tokenizer(mid)
    if use_4bit:
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        m = AutoModelForCausalLM.from_pretrained(
            mid,
            quantization_config=bnb,
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        m = AutoModelForCausalLM.from_pretrained(
            mid,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
            trust_remote_code=True,
        )
        if not torch.cuda.is_available():
            m = m.to("cpu")
    m.config.use_cache = False
    return tok, m

MODEL_ID = None
model = None
tokenizer = None
USED_4BIT = False
last_err = None

for mid in CANDIDATE_MODELS:
    for use_4bit in (True, False):
        mode = "4bit" if use_4bit else "fp16"
        try:
            print(f"Trying {mid} ({mode}) ...")
            tokenizer, model = try_load(mid, use_4bit=use_4bit)
            MODEL_ID = mid
            USED_4BIT = use_4bit
            print(f"Loaded {MODEL_ID} via {mode}")
            break
        except Exception as e:
            last_err = e
            print(f"Failed {mid} ({mode}):", e)
    if model is not None:
        break

if model is None:
    raise RuntimeError(
        "Could not load any model. Ensure Accelerator=GPU T4, Internet=On, "
        "then Restart session after cell 1.\n"
        f"Last error: {last_err}"
    )

print("READY:", MODEL_ID, "| 4bit=", USED_4BIT, "| device=", next(model.parameters()).device)

In [ ]:
# --- 6. LoRA + SFTTrainer ---
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

if USED_4BIT:
    model = prepare_model_for_kbit_training(model)

def to_text(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_text = train_ds.map(to_text, remove_columns=train_ds.column_names)
print(train_text[0]["text"][:200])

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

sft_args = SFTConfig(
    output_dir=str(WORK / "sft_checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=20,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    max_seq_length=512,
    packing=False,
    dataset_text_field="text",
    report_to=[],
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_text,
    tokenizer=tokenizer,
    peft_config=peft_config,
)
trainer.train()
trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("Saved adapter to", ADAPTER_DIR)

In [ ]:
# --- 7. Inference on eval UIDs ---
from tqdm.auto import tqdm

infer_model = trainer.model
infer_model.eval()
predictions = []

for _, r in tqdm(eval_df.iterrows(), total=len(eval_df)):
    messages = [
        {"role": "system", "content": INSTRUCTION},
        {"role": "user", "content": f"Indication: {(r['indication'] or 'Not provided').strip()}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(infer_model.device)
    with torch.no_grad():
        out = infer_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if "FINDINGS:" not in gen.upper():
        gen = "FINDINGS:\n" + gen
    predictions.append({"uid": str(r["uid"]), "prediction": gen})

pred_path = WORK / "lora_predictions.jsonl"
with open(pred_path, "w", encoding="utf-8") as f:
    for row in predictions:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

meta = {"model_id": MODEL_ID, "n": len(predictions), "used_4bit": USED_4BIT}
(WORK / "lora_run_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("Wrote", pred_path, "n=", len(predictions))
print(predictions[0]["prediction"][:400])

In [ ]:
# --- 8. Zip for download ---
import shutil

bundle = WORK / "lora_bundle"
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()
shutil.copy2(WORK / "lora_predictions.jsonl", bundle / "lora_predictions.jsonl")
shutil.copy2(WORK / "lora_run_meta.json", bundle / "lora_run_meta.json")
shutil.copytree(ADAPTER_DIR, bundle / "lora_adapter")
bundle_zip = shutil.make_archive(str(WORK / "adre_cxr_lora_bundle"), "zip", str(bundle))
print("Download:", bundle_zip)
print("Or download /kaggle/working/lora_predictions.jsonl")